# Adaptive NLI Classification System
## Multi-Genre Natural Language Inference with LLMs

This notebook presents a comprehensive analysis of Natural Language Inference (NLI) using Large Language Models with an **Adaptive AI Gatekeeper** for cost-effective inference routing.

### Research Questions
1. **Primary**: How does LLM performance on Multi-Genre NLI evolve across Zero-shot, One-shot, Few-shot, and Chain-of-Thought prompting?
2. Which of the 10 MultiNLI genres see the most significant accuracy boost with CoT?
3. Does CoT specifically reduce the "Neutral-Entailment" confusion?
4. What is the cost-to-accuracy trade-off of using reasoning-heavy prompts?

---

## 1. Setup and Imports

In [ ]:
# Standard imports
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Add project to path
sys.path.insert(0, '..')

# Project imports
from src.data import MultiNLILoader, DataPreprocessor, FeatureExtractor
from src.prompts import PromptTemplates, PromptManager
from src.llm import MockGeminiClient, BatchProcessor
from src.llm.gemini import create_client
from src.gatekeeper import DifficultyClassifier, AdaptiveRouter
from src.evaluation import MetricsCalculator, CostAnalyzer, Visualizer

print("✓ All imports successful!")
print(f"Working directory: {os.getcwd()}")

## 2. Dataset Exploration

We use the **MultiNLI** (Multi-Genre Natural Language Inference) dataset, which contains ~433,000 sentence pairs across 10 diverse genres.

In [ ]:
# Load the dataset
loader = MultiNLILoader(data_dir='../data/raw')
df_raw = loader.load_dataset('validation_matched')

print(f"Dataset size: {len(df_raw):,} samples")
print(f"Unique genres: {df_raw['genre'].nunique()}")
print(f"\nGenres: {', '.join(df_raw['genre'].unique())}")

In [ ]:
# Genre and label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genre distribution
genre_counts = df_raw['genre'].value_counts()
axes[0].barh(genre_counts.index, genre_counts.values, color='steelblue')
axes[0].set_xlabel('Number of Samples')
axes[0].set_title('Samples per Genre', fontweight='bold')

# Label distribution
label_colors = {'entailment': '#27ae60', 'neutral': '#3498db', 'contradiction': '#e74c3c'}
label_counts = df_raw['label'].value_counts()
axes[1].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
           colors=[label_colors.get(l, 'gray') for l in label_counts.index])
axes[1].set_title('Label Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Sample examples from different genres
print("Sample NLI Examples:")
print("=" * 80)

for genre in ['fiction', 'government', 'telephone']:
    sample = df_raw[df_raw['genre'] == genre].sample(1).iloc[0]
    print(f"\n[{genre.upper()}]")
    print(f"Premise: {sample['premise'][:150]}...")
    print(f"Hypothesis: {sample['hypothesis']}")
    print(f"Label: {sample['label']}")
    print("-" * 80)

## 3. Data Preprocessing

We create a stratified sample with **50 samples per genre** (500 total) to ensure robust testing across all domains.

In [ ]:
# Preprocess and sample
preprocessor = DataPreprocessor(output_dir='../data/processed')
datasets = preprocessor.process_and_save(
    df_raw,
    samples_per_genre=50,
    test_size=0.3,
    random_state=42
)

df_full = datasets['full']
df_train = datasets['train']
df_test = datasets['test']

print(f"\nFinal dataset sizes:")
print(f"  Full: {len(df_full)} samples")
print(f"  Train: {len(df_train)} samples (for gatekeeper)")
print(f"  Test: {len(df_test)} samples (for evaluation)")

In [ ]:
# Verify stratification
strat_check = pd.crosstab(df_full['genre'], df_full['label'])
print("Stratification Check (samples per genre × label):")
display(strat_check)

## 4. Feature Analysis

We extract linguistic features for the Adaptive Gatekeeper:
- **Sentence Length**: Word count of premise + hypothesis
- **Negation Count**: Presence of negation words
- **Lexical Overlap**: Jaccard similarity between premise and hypothesis

In [ ]:
# Extract features
extractor = FeatureExtractor()
df_features = extractor.extract_features_batch(df_full)

print("Extracted Features:")
print(df_features[extractor.get_feature_names()].describe().round(2))

In [ ]:
# Feature distributions by genre
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Combined length by genre
df_features.boxplot(column='combined_length', by='genre', ax=axes[0,0])
axes[0,0].set_title('Sentence Length by Genre', fontweight='bold')
axes[0,0].set_xlabel('')
plt.sca(axes[0,0])
plt.xticks(rotation=45, ha='right')

# Negations by genre
negation_by_genre = df_features.groupby('genre')['total_negations'].mean().sort_values(ascending=False)
axes[0,1].barh(negation_by_genre.index, negation_by_genre.values, color='coral')
axes[0,1].set_xlabel('Average Negation Count')
axes[0,1].set_title('Negation Frequency by Genre', fontweight='bold')

# Jaccard similarity by label
df_features.boxplot(column='jaccard_similarity', by='label', ax=axes[1,0])
axes[1,0].set_title('Lexical Overlap by Label', fontweight='bold')
axes[1,0].set_xlabel('')

# Correlation heatmap
feature_corr = df_features[extractor.get_feature_names()].corr()
sns.heatmap(feature_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1,1])
axes[1,1].set_title('Feature Correlations', fontweight='bold')

plt.suptitle('', y=1.02)
plt.tight_layout()
plt.show()

## 5. Prompt Strategy Comparison

We test four prompting strategies:
1. **Zero-shot**: No examples, just instructions
2. **One-shot**: One example provided
3. **Few-shot**: 3-5 diverse examples
4. **Chain-of-Thought (CoT)**: Step-by-step reasoning

In [ ]:
# Display prompt templates
sample_premise = "The company reported record profits in the fiscal year."
sample_hypothesis = "The company made money."

print("ZERO-SHOT PROMPT:")
print("=" * 60)
print(PromptTemplates.get_zero_shot_prompt(sample_premise, sample_hypothesis)[:500])
print("...")

In [ ]:
print("\nCHAIN-OF-THOUGHT PROMPT:")
print("=" * 60)
print(PromptTemplates.get_cot_prompt(sample_premise, sample_hypothesis)[:800])
print("...")

In [ ]:
# Run inference with mock LLM (for demonstration)
# Set use_mock=False and provide API key for real inference

USE_MOCK = True  # Change to False for real API

llm_client = create_client(use_mock=USE_MOCK, random_seed=42)
processor = BatchProcessor(llm=llm_client, batch_size=10)
prompt_manager = PromptManager()

strategies = ['zero-shot', 'one-shot', 'few-shot', 'cot']
results = {}

for strategy in strategies:
    print(f"\nRunning {strategy.upper()}...")
    
    def make_prompt(item, strat=strategy):
        return prompt_manager.create_prompt(strat, item['premise'], item['hypothesis'])
    
    result_df = processor.process_dataframe(
        df=df_test.copy(),
        prompt_fn=make_prompt,
        strategy=strategy,
        show_progress=True
    )
    results[strategy] = result_df
    
    # Quick accuracy
    valid = result_df['predicted_label'].notna()
    acc = (result_df.loc[valid, 'label'] == result_df.loc[valid, 'predicted_label']).mean()
    print(f"  Accuracy: {acc:.4f}")

print("\n✓ Inference complete!")

## 6. Classification Metrics

We evaluate using:
- **Accuracy**: Overall correct predictions
- **Macro-F1**: Unweighted mean of per-class F1 scores
- **Per-class metrics**: Precision, Recall, F1 for each label

In [ ]:
# Calculate metrics for all strategies
metrics_calc = MetricsCalculator()
all_metrics = {}

for strategy, df in results.items():
    metrics = metrics_calc.calculate_metrics_from_df(df)
    all_metrics[strategy] = metrics

# Create comparison dataframe
comparison_df = metrics_calc.compare_strategies(results)
display(comparison_df.style.background_gradient(subset=['accuracy', 'macro_f1'], cmap='Greens'))

In [ ]:
# Visualize strategy comparison
visualizer = Visualizer(output_dir='../outputs/figures')

fig = visualizer.plot_strategy_comparison(
    comparison_df,
    metric='accuracy',
    title='Strategy Accuracy Comparison',
    save_name='strategy_comparison.png'
)
plt.show()

In [ ]:
# Prompt progression (0-shot → CoT)
accuracy_progression = {s: all_metrics[s]['accuracy'] for s in strategies}
fig = visualizer.plot_prompt_progression(
    accuracy_progression,
    title='Accuracy Progression: Zero-shot → CoT',
    save_name='prompt_progression.png'
)
plt.show()

# Calculate improvement
improvement = (accuracy_progression['cot'] - accuracy_progression['zero-shot']) / accuracy_progression['zero-shot'] * 100
print(f"\nCoT improvement over Zero-shot: +{improvement:.1f}%")

## 7. Error Analysis: Neutral-Entailment Confusion

**Research Question**: Does CoT specifically reduce the "Neutral-Entailment" confusion?

In [ ]:
# Analyze confusion patterns for each strategy
confusion_analysis = {}

for strategy in strategies:
    confusion = metrics_calc.analyze_confusion_patterns(results[strategy])
    confusion_analysis[strategy] = confusion
    
confusion_df = pd.DataFrame({
    'Strategy': strategies,
    'Total Errors': [confusion_analysis[s]['total_errors'] for s in strategies],
    'Error Rate': [confusion_analysis[s]['error_rate'] for s in strategies],
    'Neutral→Entailment': [confusion_analysis[s]['neutral_to_entailment'] for s in strategies],
    'Entailment→Neutral': [confusion_analysis[s]['entailment_to_neutral'] for s in strategies],
    'N-E Confusion Rate': [confusion_analysis[s]['neutral_entailment_confusion_rate'] for s in strategies]
})

display(confusion_df.style.background_gradient(subset=['N-E Confusion Rate'], cmap='Reds_r'))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, strategy in enumerate(strategies):
    ax = axes[idx // 2, idx % 2]
    cm = all_metrics[strategy]['confusion_matrix']
    labels = all_metrics[strategy]['confusion_matrix_labels']
    
    # Normalize
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
               xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f'{strategy.upper()} Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Per-Genre Analysis

**Research Question**: Which genres benefit most from CoT reasoning?

In [ ]:
# Calculate per-genre metrics for each strategy
genre_metrics = []

for strategy in strategies:
    per_genre = metrics_calc.calculate_per_genre_metrics(results[strategy])
    per_genre['strategy'] = strategy
    genre_metrics.append(per_genre)

genre_metrics_df = pd.concat(genre_metrics, ignore_index=True)

# Pivot for heatmap
genre_accuracy_pivot = genre_metrics_df.pivot(index='genre', columns='strategy', values='accuracy')
genre_accuracy_pivot = genre_accuracy_pivot[strategies]  # Reorder columns

In [ ]:
# Per-genre heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(genre_accuracy_pivot, annot=True, fmt='.3f', cmap='RdYlGn', 
           center=0.75, vmin=0.5, vmax=1.0)
plt.title('Accuracy by Genre and Strategy', fontsize=14, fontweight='bold')
plt.xlabel('Strategy')
plt.ylabel('Genre')
plt.tight_layout()
plt.savefig('../outputs/figures/genre_accuracy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate CoT improvement by genre
genre_accuracy_pivot['CoT_Improvement'] = genre_accuracy_pivot['cot'] - genre_accuracy_pivot['zero-shot']
genre_improvement = genre_accuracy_pivot['CoT_Improvement'].sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['green' if v > 0 else 'red' for v in genre_improvement.values]
plt.barh(genre_improvement.index, genre_improvement.values, color=colors)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('Accuracy Improvement (CoT - Zero-shot)')
plt.title('CoT Improvement by Genre', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/cot_improvement_by_genre.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 3 genres benefiting from CoT:")
for genre, improvement in genre_improvement.head(3).items():
    print(f"  {genre}: +{improvement:.1%}")

## 9. Adaptive Gatekeeper Training

We train a **Logistic Regression classifier** to predict query difficulty and route queries to cost-effective strategies.

In [ ]:
# Prepare training data
zero_shot_df = results['zero-shot']
cot_df = results['cot']

# Correctness indicators
zero_shot_correct = zero_shot_df['label'] == zero_shot_df['predicted_label']
cot_correct = cot_df['label'] == cot_df['predicted_label']

print(f"Zero-shot accuracy: {zero_shot_correct.mean():.4f}")
print(f"CoT accuracy: {cot_correct.mean():.4f}")
print(f"\nSamples where CoT helps (Zero wrong, CoT right): {(~zero_shot_correct & cot_correct).sum()}")
print(f"Samples where Zero-shot sufficient: {zero_shot_correct.sum()}")

In [ ]:
# Train difficulty classifier with cross-validation
classifier = DifficultyClassifier(n_folds=5, random_state=42)

# We need features in the dataframe
df_with_features = extractor.extract_features_batch(zero_shot_df)

train_results = classifier.fit(
    df_with_features,
    zero_shot_correct,
    cot_correct
)

print(f"\nGatekeeper CV Accuracy: {train_results['cv_accuracy']:.4f}")
print(f"Easy samples: {train_results['n_easy']} ({train_results['n_easy']/len(df_with_features)*100:.1f}%)")
print(f"Hard samples: {train_results['n_hard']} ({train_results['n_hard']/len(df_with_features)*100:.1f}%)")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': train_results['feature_importance'].keys(),
    'Importance': train_results['feature_importance'].values()
}).sort_values('Importance', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['green' if v > 0 else 'red' for v in feature_importance['Importance']]
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color=colors)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('Feature Importance (Coefficient)')
plt.title('Gatekeeper Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("  Positive values → Feature predicts HARD queries")
print("  Negative values → Feature predicts EASY queries")

In [ ]:
# Difficulty by genre
genre_difficulty = classifier.analyze_by_genre(df_with_features)
display(genre_difficulty.style.background_gradient(subset=['hard_ratio'], cmap='Reds'))

## 10. Cost-Benefit Analysis

**Research Question**: What is the ROI of the Adaptive Gatekeeper?

In [ ]:
# Cost analysis
cost_analyzer = CostAnalyzer()

# Compare strategy costs
cost_comparison = cost_analyzer.compare_strategies(n_samples=500)
display(cost_comparison.style.format({
    'cost_usd': '${:.6f}',
    'cost_per_sample': '${:.8f}',
    'relative_cost': '{:.2f}x'
}))

In [ ]:
# Create adaptive router
router = AdaptiveRouter(classifier=classifier, use_tiered_routing=True)

# Route all test samples
df_routed = router.add_routing_to_dataframe(df_with_features)

# Routing distribution
routing_dist = df_routed['routed_strategy'].value_counts()
print("Adaptive Routing Distribution:")
for strategy, count in routing_dist.items():
    print(f"  {strategy}: {count} ({count/len(df_routed)*100:.1f}%)")

In [ ]:
# Calculate savings vs all-CoT baseline
strategy_distribution = routing_dist.to_dict()
savings = cost_analyzer.calculate_adaptive_savings(strategy_distribution, baseline_strategy='cot')

print("\n" + "=" * 50)
print("COST SAVINGS ANALYSIS")
print("=" * 50)
print(f"\nBaseline (all CoT): ${savings['baseline_cost_usd']:.6f}")
print(f"Adaptive routing:   ${savings['adaptive_cost_usd']:.6f}")
print(f"\nCost saved:         ${savings['cost_savings_usd']:.6f} ({savings['cost_savings_pct']:.1f}%)")
print(f"Token saved:        {savings['token_savings']:,} ({savings['token_savings_pct']:.1f}%)")

In [ ]:
# Cost-Accuracy Frontier
strategy_metrics_dict = {s: {'accuracy': all_metrics[s]['accuracy'], 'macro_f1': all_metrics[s]['macro_f1']} 
                        for s in strategies}

frontier_df = cost_analyzer.calculate_cost_accuracy_frontier(strategy_metrics_dict, n_samples=500)

fig = visualizer.plot_cost_accuracy_frontier(
    frontier_df,
    title='Cost-Accuracy Frontier (500 samples)',
    save_name='cost_accuracy_frontier.png'
)
plt.show()

print("\nPareto-optimal strategies:")
for _, row in frontier_df[frontier_df['pareto_optimal']].iterrows():
    print(f"  {row['strategy']}: Accuracy={row['accuracy']:.3f}, Cost=${row['cost_usd']:.6f}")

In [ ]:
# Budget projection (£20 budget)
budget_gbp = 20
budget_usd = budget_gbp * 1.25  # Approximate conversion

projection = cost_analyzer.project_budget_usage(budget_usd)

print(f"\nBudget Projection (£{budget_gbp} ≈ ${budget_usd:.2f}):")
print(f"  Max samples with equal distribution: {projection['max_samples']:,}")
print(f"  Average cost per sample: ${projection['avg_cost_per_sample']:.8f}")

## 11. Summary Dashboard

In [ ]:
# Create comprehensive dashboard
confusion_matrices = {s: all_metrics[s]['confusion_matrix'] for s in strategies}

fig = visualizer.create_summary_dashboard(
    comparison_df,
    confusion_matrices,
    frontier_df,
    save_name='summary_dashboard.png'
)
plt.show()

## 12. Conclusions and Limitations

### Key Findings

1. **Prompt Progression**: Performance improves from Zero-shot → CoT, with Few-shot providing a good balance.

2. **Neutral-Entailment Confusion**: CoT reduces this common error pattern.

3. **Genre Variance**: Some genres (e.g., government, nineeleven) benefit more from CoT due to complex language.

4. **Adaptive Gatekeeper**: The routing strategy can reduce costs while maintaining accuracy.

### Limitations

1. **Mock LLM**: Results shown here use a simulated LLM. Real API results may differ.

2. **Sample Size**: 50 samples per genre may not capture full genre variance.

3. **Single Model**: Only tested with Gemini API; other models may perform differently.

4. **Gatekeeper Generalization**: Trained on validation data; may not generalize perfectly to new domains.

In [ ]:
# Export final results
final_results = {
    'strategy_comparison': comparison_df.to_dict('records'),
    'best_strategy': comparison_df.iloc[0]['strategy'],
    'best_accuracy': comparison_df.iloc[0]['accuracy'],
    'cost_savings_pct': savings['cost_savings_pct'],
    'token_savings_pct': savings['token_savings_pct'],
    'gatekeeper_cv_accuracy': train_results['cv_accuracy']
}

import json
with open('../data/predictions/final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"\nBest Strategy: {final_results['best_strategy'].upper()}")
print(f"Best Accuracy: {final_results['best_accuracy']:.4f}")
print(f"\nAdaptive Gatekeeper:")
print(f"  CV Accuracy: {final_results['gatekeeper_cv_accuracy']:.4f}")
print(f"  Cost Savings: {final_results['cost_savings_pct']:.1f}%")
print(f"  Token Savings: {final_results['token_savings_pct']:.1f}%")
print("\n✓ Results exported to data/predictions/final_results.json")